# RAG 技术学习教程

本 Notebook 将带你一步步构建一个完整的 RAG（检索增强生成）系统。

**学习目标：**
- 理解 RAG 的每个组成步骤
- 动手运行每个代码单元，观察中间结果
- 尝试修改参数，感受不同设置的影响

**前置条件：**
- 已安装依赖：`pip install -r requirements.txt`
- 已配置 `.env` 文件（或设置环境变量 `OPENAI_API_KEY`）

## 0. 环境准备

In [ ]:
import os
import sys

# 确保能找到 rag 包
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

from dotenv import load_dotenv
load_dotenv()

# 检查 API Key 是否已设置
if os.getenv('OPENAI_API_KEY'):
    print('✅ OPENAI_API_KEY 已设置')
else:
    print('⚠️  未检测到 OPENAI_API_KEY，如需使用 OpenAI 请先配置')
    print('   也可以使用 Ollama 本地模型（免费）'

## 第一步：文档加载

RAG 的第一步是将文档加载为 LangChain 的 `Document` 对象。
每个 Document 包含：
- `page_content`：文档的文本内容
- `metadata`：元数据（来源文件、页码等）

In [ ]:
from rag.loader import load_text, load_documents

# 加载示例文档
docs = load_documents([
    '../data/rag_introduction.txt',
    '../data/langchain_intro.txt',
])

print(f'加载了 {len(docs)} 篇文档')
print(f'\n第一篇文档预览：')
print(f'来源: {docs[0].metadata["source"]}')
print(f'内容前200字:\n{docs[0].page_content[:200]}')

## 第二步：文本分割

长文档需要被分割成小块（chunk），原因：
1. 嵌入模型有 Token 限制
2. 小块更容易精确检索
3. 避免将不相关信息混入上下文

**关键参数：**
- `chunk_size`：每块的最大字符数（试试改成 200 或 1000，观察块数变化）
- `chunk_overlap`：相邻块的重叠字符数（保证语义连续性）

In [ ]:
from rag.splitter import split_documents

chunks = split_documents(
    docs,
    chunk_size=400,    # 试试改成 200 或 800
    chunk_overlap=50,
)

print(f'分割结果：{len(docs)} 篇文档 → {len(chunks)} 个块')
print(f'\n第一个块：')
print(f'内容: {chunks[0].page_content}')
print(f'元数据: {chunks[0].metadata}')
print(f'\n第二个块（注意与第一块的重叠部分）：')
print(f'内容: {chunks[1].page_content[:100]}')

## 第三步：文本嵌入

嵌入（Embedding）是将文本转换为向量的过程。
语义相近的文本，其向量在高维空间中距离更近。

这里演示嵌入模型的基本使用，实际向量库构建在下一步完成。

In [ ]:
from rag.embedder import get_embeddings

# 选择嵌入模型
# 方式一：OpenAI（需要 API Key）
embeddings = get_embeddings('openai')

# 方式二：Ollama 本地模型（需要先 ollama pull nomic-embed-text）
# embeddings = get_embeddings('ollama', model='nomic-embed-text')

# 演示：将两段文本转为向量并计算相似度
import numpy as np

texts = [
    'RAG 是检索增强生成技术',
    '检索增强生成是一种 AI 技术',  # 语义相近
    '今天天气很好',                 # 语义不相关
]

vectors = embeddings.embed_documents(texts)
print(f'向量维度: {len(vectors[0])}')

def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sim_12 = cosine_similarity(vectors[0], vectors[1])
sim_13 = cosine_similarity(vectors[0], vectors[2])
print(f'\n句子1 vs 句子2（语义相近）的余弦相似度: {sim_12:.4f}')
print(f'句子1 vs 句子3（语义不相关）的余弦相似度: {sim_13:.4f}')
print('\n✅ 语义相近的句子相似度更高，这就是向量检索的基础！')

## 第四步：构建向量库

将所有文档块的向量存入 ChromaDB 向量数据库。

In [ ]:
from rag.retriever import build_vector_store, get_retriever

# 构建向量库（内存模式，重启后数据丢失）
vector_store = build_vector_store(
    documents=chunks,
    embeddings=embeddings,
    persist_dir=None,  # 改为路径如 '/tmp/my_chroma' 可持久化
)

print('向量库构建完成！')

## 第五步：检索

输入查询，检索最相关的文档块。

**尝试：** 修改 `query` 和 `k` 的值，观察检索结果的变化。

In [ ]:
retriever = get_retriever(vector_store, k=3)

query = 'RAG 解决了什么问题？'
docs = retriever.invoke(query)

print(f'查询: "{query}"')
print(f'检索到 {len(docs)} 个相关块：\n')

for i, doc in enumerate(docs, 1):
    print(f'--- 块 {i} ---')
    print(f'来源: {doc.metadata.get("source", "未知")}')
    print(f'内容: {doc.page_content[:200]}')
    print()

## 第六步：生成答案

将检索到的文档块作为上下文，让 LLM 基于上下文回答问题。

In [ ]:
from rag.generator import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 初始化 LLM
llm = get_llm('openai', model='gpt-4o-mini')
# llm = get_llm('ollama', model='qwen2.5')

# 构建提示词
prompt = ChatPromptTemplate.from_messages([
    ('system', '请根据以下参考资料回答用户问题，如果资料中没有相关信息请说明。\n\n参考资料:\n{context}'),
    ('human', '{question}'),
])

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# 构建 RAG 链
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 提问
question = 'RAG 解决了大语言模型的哪些问题？'
answer = rag_chain.invoke(question)

print(f'问题: {question}')
print(f'\n答案:')
print(answer)

## 完整流水线

上面的步骤已封装在 `RAGPipeline` 类中，可以直接使用：

In [ ]:
from rag import RAGPipeline

# 初始化流水线
pipeline = RAGPipeline(
    embeddings=get_embeddings('openai'),
    llm=get_llm('openai', model='gpt-4o-mini'),
    chunk_size=400,
    chunk_overlap=50,
    retrieval_k=3,
)

# 构建知识库
pipeline.build([
    '../data/rag_introduction.txt',
    '../data/langchain_intro.txt',
])

# 连续提问
questions = [
    'LangChain 是什么？',
    'RAG 和微调有什么区别？',
    'ChromaDB 有什么特点？',
]

for q in questions:
    print(f'\n{'='*50}')
    pipeline.ask(q)

## 🎯 练习题

完成以下练习，加深对 RAG 的理解：

1. **调整 chunk_size**：试试 100、500、1000，观察检索质量如何变化
2. **加载自己的文档**：将你感兴趣的文档放入 `data/` 目录并加载
3. **修改提示词**：尝试让 LLM 以不同风格回答（如：严谨学术风格、口语化风格）
4. **对比检索方式**：将 `search_type` 从 `similarity` 改为 `mmr`，观察结果多样性
5. **实现持久化**：将 `persist_dir` 设为某个目录，重启后验证可以直接加载